In [1]:
# ─────────────────────────────────────────────
# CELL 1: Install dependencies
# ─────────────────────────────────────────────
!pip install -q transformers bitsandbytes accelerate snac soundfile torch

# ─────────────────────────────────────────────
# CELL 2: Inference
# ─────────────────────────────────────────────
import torch
import soundfile as sf
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from snac import SNAC

# ── CONFIG ───────────────────────────────────
MODEL_ID    = "canopylabs/orpheus-3b-0.1-ft"   # ← MUST be ft, not pretrained
SPEAKER_TAG = "tara"
TEXT        = "Hello, this is Orpheus speaking."
OUTPUT_WAV  = "output.wav"

DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS = 1200
TEMPERATURE    = 0.6
TOP_P          = 0.95
REP_PENALTY    = 1.1

# Special token IDs (fixed for this model)
START_TOKEN = 128257   # <custom_token_1> — marks start of audio tokens
END_TOKEN   = 128258   # <custom_token_2> — marks end of audio tokens
SNAC_SR     = 24000

# ── Load tokenizer & model ───────────────────
print(f"Loading {MODEL_ID} ...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()
print("✅ Model ready.")

# ── Load SNAC decoder ────────────────────────
print("Loading SNAC ...")
snac_model = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").eval().to(DEVICE)
print("✅ SNAC ready.")

# ── Build prompt & generate ──────────────────
# Correct format: "speaker: text"  — NO <|audio|> wrapper
prompt = f"{SPEAKER_TAG}: {TEXT}"
print(f"\nPrompt: {prompt!r}")

inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

print("Generating audio tokens ...")
with torch.no_grad():
    generated_ids = model.generate(
        input_ids      = inputs.input_ids,
        attention_mask = inputs.attention_mask,
        max_new_tokens = MAX_NEW_TOKENS,
        do_sample      = True,
        temperature    = TEMPERATURE,
        top_p          = TOP_P,
        repetition_penalty = REP_PENALTY,
        num_return_sequences = 1,
        eos_token_id   = END_TOKEN,
        pad_token_id   = tokenizer.eos_token_id,
    )

# ── Extract audio tokens between START and END markers ───
output_ids = generated_ids[0].tolist()

# Find where audio tokens begin
try:
    start_idx = output_ids.index(START_TOKEN) + 1
except ValueError:
    # Fallback: skip the input prompt tokens
    start_idx = inputs.input_ids.shape[1]

# Strip everything before start and remove the end token
audio_token_ids = [
    t for t in output_ids[start_idx:]
    if t != END_TOKEN and t != tokenizer.eos_token_id
]

print(f"Raw audio tokens extracted: {len(audio_token_ids)}")

# ── Convert token IDs → SNAC codes ──────────
# Tokens are offset from 128266, grouped in 7s per frame
AUDIO_OFFSET = 128266
LAYER_SIZE   = 4096

# Keep only valid audio-range tokens
audio_token_ids = [
    t for t in audio_token_ids
    if AUDIO_OFFSET <= t < AUDIO_OFFSET + 7 * LAYER_SIZE
]

frames = len(audio_token_ids) // 7
print(f"Frames: {frames}  |  Est. duration: {frames / 75:.2f}s")

if frames == 0:
    raise RuntimeError("No audio frames decoded. Try increasing MAX_NEW_TOKENS or check the model.")

audio_token_ids = audio_token_ids[: frames * 7]

l0, l1, l2 = [], [], []
for i in range(frames):
    base = i * 7
    c = [
        max(0, min(4095, audio_token_ids[base + j] - AUDIO_OFFSET - j * LAYER_SIZE))
        for j in range(7)
    ]
    l0.append(c[0])
    l1.extend([c[1], c[4]])
    l2.extend([c[2], c[3], c[5], c[6]])

codes = [
    torch.tensor(l0, dtype=torch.long, device=DEVICE).unsqueeze(0),
    torch.tensor(l1, dtype=torch.long, device=DEVICE).unsqueeze(0),
    torch.tensor(l2, dtype=torch.long, device=DEVICE).unsqueeze(0),
]

# ── Decode to waveform ───────────────────────
with torch.no_grad():
    waveform = snac_model.decode(codes)

audio_np = waveform.squeeze().cpu().numpy()
sf.write(OUTPUT_WAV, audio_np, SNAC_SR)
print(f"✅ Saved → '{OUTPUT_WAV}'  ({len(audio_np) / SNAC_SR:.2f}s @ {SNAC_SR} Hz)")

# ── Play in notebook ─────────────────────────
from IPython.display import Audio, display
display(Audio(audio_np, rate=SNAC_SR))

Loading canopylabs/orpheus-3b-0.1-ft ...


Loading weights:   0%|          | 0/255 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ Model ready.
Loading SNAC ...
✅ SNAC ready.

Prompt: 'tara: Өнөөдөр гадаа хасах хорин хэм хүйтэн байна.'
Generating audio tokens ...
Raw audio tokens extracted: 1196
Frames: 170  |  Est. duration: 2.27s
✅ Saved → 'output.wav'  (14.51s @ 24000 Hz)
